# Vilex — Kaggle Stage 5 Only (OmniVoice TTS)

**Repo:** https://github.com/thaiphu05/Vilex @ `dev-thai`  
**Input:** Stage 1–4 JSONs (run locally → upload as a Kaggle Dataset)  
**Output:** `data/vi_audio/**/dialogues/dialogue.wav` (stereo ch0=assistant ch1=user) + `meta.json`

Stage 5 reads `config.yaml`; this notebook overrides it through `VILEX_*` env vars (see `docs/CONFIGURATION.md`).

## Kaggle Settings (do once)
- `Settings → Internet ON` (for `pip install git+OmniVoice`)
- `Settings → Accelerator → GPU T4 x2`
- `Add Input`: mount (1) the Stage 1–4 dialogues (`data/vi_tt_bc`) and (2) the `voice_clone` pool
- No `Secrets` needed (Stage 5 makes no Gemini calls); no kernel restart (single env)

In [ ]:
# Cell 1 — Config helpers (edit BRANCH to match your fork)
BRANCH = "dev-thai"
REPO = "https://github.com/thaiphu05/Vilex.git"

In [ ]:
# Cell 2 — Clone code (data comes from Kaggle Datasets)
!git clone -b $BRANCH $REPO
%cd Vilex
!git branch --show-current && git log --oneline -3 && ls -lh

In [ ]:
# Cell 3 — Install Stage 5 deps (OmniVoice)
!pip install -q -r requirements-stage5.txt
!pip install -q git+https://github.com/k2-fsa/OmniVoice.git
!python -c "import torch; print(f'torch={torch.__version__} cuda={torch.cuda.is_available()}')"
!python -c "import whisperx, silero_vad, yaml; print('whisperx+silero+yaml ok')"

In [ ]:
# Stage 5 config — discover input dialogues + voice pool, then override config.yaml.
import os, json, glob

VOICE_POOL = "/kaggle/input/voice-clone"   # <- your voice-clone dataset mount
for cand in [VOICE_POOL, "/kaggle/input/voice_clone", "voice_clone",
             "/kaggle/working/voice_clone"]:
    if os.path.isdir(cand):
        VOICE_POOL = cand
        break

# Stage-4b output: local data/vi_tt_bc if it persisted, else a Kaggle Dataset mount.
BC_ROOT = "data/vi_tt_bc"
if not glob.glob(os.path.join(BC_ROOT, "**", "text_dialogue_*", "*", "*.json"), recursive=True):
    for root in sorted(glob.glob("/kaggle/input/*")):
        if glob.glob(os.path.join(root, "**", "text_dialogue_*", "*", "*.json"), recursive=True):
            BC_ROOT = root
            break

def set_vi(overrides):
    for key, val in overrides.items():
        if isinstance(val, bool):
            sval = "true" if val else "false"
        elif isinstance(val, (list, dict)):
            sval = json.dumps(val, ensure_ascii=False)
        else:
            sval = str(val)
        os.environ["VILEX_" + key.upper().replace(".", "__")] = sval

set_vi({
    "paths.bc_root": BC_ROOT,               # recursive glob: any depth under it
    "paths.audio_root": "data/vi_audio",
    "paths.voice_clone_pool": VOICE_POOL,
    "stage5_tts.backend": "omnivoice",
    "stage5_tts.language": "vi",
    "stage5_tts.device": "cuda",            # OOM -> "cpu"
    "stage5_tts.num_variants": 1,
    "stage5_tts.max_dialogues": 1,          # 0 = all
    "stage5_tts.tags.render": True,         # keep [laughter]/[sigh]/... tags
})

wavs = sorted(glob.glob(os.path.join(VOICE_POOL, "*.wav")))
missing = [w for w in wavs if not os.path.isfile(os.path.splitext(w)[0] + ".txt")]
print(f"VOICE_POOL={VOICE_POOL} ({len(wavs)} wavs, {len(missing)} missing .txt)")
print(f"BC_ROOT={BC_ROOT}")
if len(wavs) < 2:
    print("ERROR: voice pool needs >=2 wavs with sidecar .txt")

In [ ]:
# Cell 4 — Stage 5: OmniVoice render
!python tts_render/convert_spoken.py

In [ ]:
# Cell 5 — If CUDA OOM, rerun with CPU
import os
os.environ["VILEX_STAGE5_TTS__DEVICE"] = "cpu"
!python tts_render/convert_spoken.py

In [ ]:
# Preview + export
!find data/vi_audio -type f 2>/dev/null | head -30
try:
    from IPython.display import Audio, display
    import glob as _g
    wavs = _g.glob("data/vi_audio/**/dialogue.wav", recursive=True)
    if wavs:
        print(f"Preview: {wavs[0]}")
        display(Audio(wavs[0]))
    else:
        print("No dialogue.wav yet — check logs above")
except Exception as e:
    print(e)
!zip -qr /kaggle/working/vi_audio.zip data/vi_audio
!ls -lh /kaggle/working/vi_audio.zip 2>/dev/null

## Notes
- **Local export of Stage 1–4 output:** `zip -r vilex-phase1-dialogues.zip data/vi_tt_bc` → `Kaggle → Datasets → New` (repo ignores `outputs/`).
- **Full run:** set `stage5_tts.max_dialogues: 0` and raise `num_variants` (these `VILEX_*` overrides live in the Stage 5 config cell).
- **Override anything:** `VILEX_STAGE5_TTS__DEVICE=cuda`, `VILEX_STAGE5_TTS__TAGS__RENDER=true`, ... — see `docs/CONFIGURATION.md`.
- **Output layout:** `data/vi_audio/text_dialogue_<ds>/<split>/<id>/varNN/dialogues/dialogue.wav` + `meta.json`.